# Bounded swarm expert iteration

**Run this notebook on a Google Colab A100 40 GB or 80 GB runtime.** In VS Code, open this notebook, choose **Select Kernel → Colab**, select the A100 runtime, then choose **Run All**. The locked dataset is included in the repository, so this run does not require an API key.

The notebook clones the supporting code into `/content/swarm-expert-iter` and runs each training and evaluation stage in sequence. All GPU work runs in subprocesses. The committed seed plans and tasks were generated with DeepSeek. Gen-0 and Gen-1 each make one greedy local attempt on the same 10 held-out tasks.

At the end, open **Colab → Contents → swarm-expert-iter** in VS Code and use **Download…** on `handoff_bundle.zip`. The archive includes the video, measured scorecard, metrics, adapters, and repository source.

[Google Colab setup in VS Code](https://developers.googleblog.com/en/google-colab-is-coming-to-vs-code/) · [Deploy DeepSeek model](https://deployai.readme.io/docs/deepseek)

## 0 · Configuration and dataset lock

The repository contains the locked dataset. The first cell checks its manifest and file checksums before installing the training environment.

In [ ]:
from pathlib import Path
import os
import platform
import signal
import subprocess
import sys

RUN_DIR = Path("/content/swarm-expert-iter")
REPOSITORY_URL = "https://github.com/HamzaDevv/doom_escape.git"

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Select the Google Colab A100 kernel before running this notebook.")
try:
    gpu_names = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True, stderr=subprocess.DEVNULL, timeout=10,
    ).strip().splitlines()
except (OSError, subprocess.SubprocessError) as error:
    raise RuntimeError("Select a Google Colab A100 GPU runtime.") from error
if len(gpu_names) != 1 or "A100" not in gpu_names[0] or "MIG" in gpu_names[0]:
    raise RuntimeError(f"Expected one full A100 GPU. Found {gpu_names}.")

RUN_DIR = RUN_DIR.expanduser().resolve()
if not RUN_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(RUN_DIR)], check=True)
else:
    if not (RUN_DIR / ".git").is_dir():
        raise RuntimeError(f"Choose a new run directory, or use the existing cloned repository: {RUN_DIR}")
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=RUN_DIR, text=True).strip()
    if origin.removesuffix(".git") != REPOSITORY_URL.removesuffix(".git"):
        raise RuntimeError(f"Unexpected repository at {RUN_DIR}: {origin}")
(RUN_DIR / ".swarm-project").touch(exist_ok=True)
LOG_DIR = RUN_DIR / "workspace" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
PYTHON = str(RUN_DIR / ".venv/bin/python")

subprocess.run(
    [sys.executable, "-c", "from swarm_common import load_datasets; load_datasets()"],
    cwd=RUN_DIR, check=True,
)
print(f"Locked dataset verified | {gpu_names[0]} | {RUN_DIR}")

In [ ]:
def run_stage(label, *command, environment=None, progress_seconds=25, announce=None):
    env = {key: value for key, value in os.environ.items() if not key.startswith(("DEEPSEEK_", "DEPLOY_"))}
    if environment:
        env.update(environment)
    env.update(PYTHONUNBUFFERED="1", SWARM_BOOTSTRAP_PYTHON=sys.executable,
               HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false",
               TRANSFORMERS_VERBOSITY="error", TQDM_DISABLE="1", WANDB_MODE="disabled")
    log = LOG_DIR / f"{label}.log"
    print(f"Running {label} • {log}", flush=True)
    with log.open("wb") as stream:
        process = subprocess.Popen(
            command, cwd=RUN_DIR, env=env, stdin=subprocess.DEVNULL, stdout=stream,
            stderr=subprocess.STDOUT, start_new_session=True,
        )
        try:
            while True:
                try:
                    status = process.wait(timeout=progress_seconds)
                    break
                except subprocess.TimeoutExpired:
                    print(f"{label} is running.", flush=True)
        except KeyboardInterrupt:
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=40)
            except ProcessLookupError:
                pass
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            raise
    if status:
        if label == "seed-data":
            print("Synthesis stopped. Inspect the HTTP status or failing dataset slot in its log.")
        else:
            with log.open("rb") as stream:
                stream.seek(max(0, log.stat().st_size - 5000))
                print(stream.read().decode("utf-8", errors="replace"))
        raise RuntimeError(f"{label} exited with status {status}. See {log}.")
    if announce:
        with log.open("rb") as stream:
            stream.seek(max(0, log.stat().st_size - 3000))
            print(stream.read().decode("utf-8", errors="replace").strip())
    else:
        print(f"{label} complete.")

## 1 · Install and verify the environment

Setup installs the Python stack in a venv, the headless recording tools, and the Linux seccomp runtime. An isolated subprocess self-test must pass before any generated code runs.

In [ ]:
run_stage("environment", "bash", "setup_env.sh")

## 2 · Verify the locked dataset and fit the seed LoRA

The notebook validates the committed dataset against its locked checksums and trains the seed adapter on the 80 coordinator plans. The 10 held-out tasks stay outside seed training.

In [ ]:
run_stage("seed-data", PYTHON, "generate_seed_data.py")
run_stage("seed-lora", PYTHON, "seed_lora.py", announce=True)

## 3 · Smoke-test training and hot-swap

A one-step continuation checks the trainer's memory budget. A live worker serves inference before and after an in-place swap, then exits.

In [ ]:
run_stage("worker-trainer-smoke", PYTHON, "swarm_runtime.py", announce=True)

## 4 · Record the full run

Prime starts three independent workers on ports 8001–8003. It runs Gen-0, collects 24 verified exploration traces, launches the continuation trainer while the mesh serves requests, updates the live adapters, and runs Gen-1. The recorder stores the terminal HUD without streaming the child logs into it. This is the longest step.

In [ ]:
run_stage("paired-benchmark", PYTHON, "record_run.py", announce=True)

## 5 · Render and validate the proof-of-work video

The measured scorecard is appended as a four-second outro. The video is decoded and checked for resolution, frame rate, size, and duration before it is promoted to the artifacts directory.

In [ ]:
run_stage("scorecard", PYTHON, "plot_results.py")
run_stage("video", "bash", "render_demo_video.sh", announce=True)
from IPython.display import Image, display
display(Image(filename=str(RUN_DIR / "artifacts/benchmark_scorecard.png"), width=960))

## 6 · Commit and package the handoff

The README uses the recorded scores. The archive is still produced if the improvement gate is unmet, with the measured result labeled in the README, metrics and terminal recording.

Download the archive from the VS Code Colab **Contents** view. A new `RUN_DIR` starts a new experiment. Rerunning the same directory reuses completed datasets and checkpoints. The generated Git commit stays in Colab, and the archive contains its tracked files.

In [ ]:
run_stage("handoff", PYTHON, "package_handoff.py", announce=True)